# ML-04 - Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** - each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

- One row = one content item, which is one page. Why: the dictionary says each `content_id` is unique in this starter slice, so the decision lands on a page.
- Time window = the trailing 90 days at export time. Why: the traffic fields here are all 90-day aggregates, so pages are being compared on the same measurement window.
- Features = `avg_position`, `content_age_days`, `search_volume`, `content_type`, and `main_intent`. Why: these are knowable when I score the queue and they help explain why CTR differs across pages.
- Label / proxy = `ctr`. Why: CTR is the observed click outcome I want to rank opportunity around, so it belongs on the right-hand side, not in the inputs.
- Output = a ranked review queue for pages with at least 1,000 impressions. Why: editors need a short action list, not a claim that the model can guarantee lift.

In [1]:
import pandas as pd

csv_path = "D:/Flyrank/Flyrank-assignments/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)
scoring_slice = df[df["impressions_90d"] >= 1000].copy()

contract_answers = pd.DataFrame(
    [
        {
            "contract_line": "One row = one page",
            "plain_words_answer": "Each row is one content item identified by one content_id.",
            "why": "The final ranking is page-level, so the unit of analysis should be page-level too.",
        },
        {
            "contract_line": "Window = trailing 90 days",
            "plain_words_answer": "The metrics come from one trailing 90-day snapshot.",
            "why": "A shared window keeps the comparisons fair across pages.",
        },
        {
            "contract_line": "Features = five safe inputs",
            "plain_words_answer": "I will use avg_position, content_age_days, search_volume, content_type, and main_intent.",
            "why": "They are available at scoring time and help explain expected CTR differences.",
        },
        {
            "contract_line": "Label / proxy = ctr",
            "plain_words_answer": "CTR is the observed outcome I am trying to rank against.",
            "why": "Using ctr as the answer keeps the task tied to a measured click behavior.",
        },
        {
            "contract_line": "Output = ranked queue",
            "plain_words_answer": "The notebook should hand editors a prioritized list of visible pages to review.",
            "why": "That matches the real decision from ML-02 and ML-03: who to review first.",
        },
    ]
)

contract_answers

                 contract_line  ...                                                why
0           One row = one page  ...  The final ranking is page-level, so the unit o...
1    Window = trailing 90 days  ...  A shared window keeps the comparisons fair acr...
2  Features = five safe inputs  ...  They are available at scoring time and help ex...
3          Label / proxy = ctr  ...  Using ctr as the answer keeps the task tied to...
4        Output = ranked queue  ...  That matches the real decision from ML-02 and ...

[5 rows x 3 columns]


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
field_buckets = pd.DataFrame(
    [
        {"bucket": "feature", "field": "avg_position", "why": "CTR expectations depend heavily on rank, so this is needed context before scoring."},
        {"bucket": "feature", "field": "content_age_days", "why": "Older pages can have different refresh opportunity than new pages."},
        {"bucket": "feature", "field": "search_volume", "why": "Keyword demand helps separate weak CTR on a high-value topic from weak CTR on a tiny topic."},
        {"bucket": "feature", "field": "content_type", "why": "CTR norms differ by page type, so the model needs that context."},
        {"bucket": "feature", "field": "main_intent", "why": "Search intent changes how likely users are to click."},
        {"bucket": "label/proxy", "field": "ctr", "why": "This is the observed click outcome for the ranking task, so it is the answer, not an input."},
        {"bucket": "context", "field": "content_id", "why": "Needed to identify the page in the final queue, but not a learning signal."},
        {"bucket": "context", "field": "client_id", "why": "Needed for grouped validation and client-level reading, but not a feature."},
        {"bucket": "excluded", "field": "clicks_90d", "why": "CTR is computed from clicks and impressions, so clicks_90d leaks the answer."},
        {"bucket": "excluded", "field": "impressions_90d", "why": "This is also an ingredient of ctr, so it would leak the answer when paired with clicks."},
        {"bucket": "excluded", "field": "trend_direction", "why": "It is a separate derived label family and does not belong in the CTR lane feature set."},
        {"bucket": "excluded", "field": "trend_pct", "why": "It is derived from the trend label inputs and is not needed for this scoring slice."},
    ]
)

field_buckets

         bucket  ...                                                why
0       feature  ...  CTR expectations depend heavily on rank, so th...
1       feature  ...  Older pages can have different refresh opportu...
2       feature  ...  Keyword demand helps separate weak CTR on a hi...
3       feature  ...  CTR norms differ by page type, so the model ne...
4       feature  ...  Search intent changes how likely users are to ...
5   label/proxy  ...  This is the observed click outcome for the ran...
6       context  ...  Needed to identify the page in the final queue...
7       context  ...  Needed for grouped validation and client-level...
8      excluded  ...  CTR is computed from clicks and impressions, s...
9      excluded  ...  This is also an ingredient of ctr, so it would...
10     excluded  ...  It is a separate derived label family and does...
11     excluded  ...  It is derived from the trend label inputs and ...

[12 rows x 3 columns]


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
query_1 = pd.DataFrame(
    {
        "rows": [len(df)],
        "distinct_content_id": [df["content_id"].nunique()],
        "distinct_client_id": [df["client_id"].nunique()],
        "grain_holds_is_true": [len(df) == df["content_id"].nunique()],
    }
)
print("Query 1 - grain check")
print(query_1.to_string(index=False))

query_2 = pd.DataFrame(
    {
        "min_content_age_days": [int(df["content_age_days"].min())],
        "max_days_with_impressions": [int(df["days_with_impressions"].max())],
        "max_days_with_sessions": [int(df["days_with_sessions"].max())],
        "rows_in_scoring_slice": [len(scoring_slice)],
        "window_claim_is_true": [
            (df["content_age_days"].min() >= 90)
            and (df["days_with_impressions"].max() <= 90)
            and (df["days_with_sessions"].max() <= 90)
        ],
    }
)
print("\nQuery 2 - 90 day window and slice check")
print(query_2.to_string(index=False))

query_3 = (
    df.groupby("content_type")[["search_volume", "main_intent"]]
    .apply(lambda block: block.isna().mean())
    .reset_index()
    .rename(
        columns={
            "search_volume": "search_volume_missing_rate",
            "main_intent": "main_intent_missing_rate",
        }
    )
)
print("\nQuery 3 - missingness pattern by content_type")
print(query_3.to_string(index=False))

Query 1 - grain check
 rows  distinct_content_id  distinct_client_id  grain_holds_is_true
30000                30000                  32                 True

Query 2 - 90 day window and slice check
 min_content_age_days  max_days_with_impressions  max_days_with_sessions  rows_in_scoring_slice  window_claim_is_true
                   90                         88                      90                  13512                  True

Query 3 - missingness pattern by content_type
      content_type  search_volume_missing_rate  main_intent_missing_rate
comparison article                    0.000000                  0.000000
    feedly article                    1.000000                  1.000000
   keyword article                    0.013673                  0.010218


## 4. Data limits

- Named limitation: same-window snapshot limitation. Why: this CSV is one trailing 90-day export, so it can rank pages inside this moment but it cannot prove what will happen in a future month.
- Keyword context is not complete for every page type. Why: `feedly article` rows are missing keyword fields by design, so missingness is patterned rather than random.
- This slice supports decision-support, not causation. Why: the notebook shows observed differences in CTR, but it cannot prove that a refresh alone causes the lift.

In [4]:
limits = pd.DataFrame(
    [
        {
            "limitation_name": "same-window snapshot limitation",
            "evidence": "All rows come from one trailing 90-day export rather than a forward holdout timeline.",
            "why_it_matters": "I can rank current opportunity, but I cannot claim future lift from this slice alone.",
        },
        {
            "limitation_name": "patterned keyword missingness",
            "evidence": "Feedly rows have 100% missing search_volume and main_intent in the verification query.",
            "why_it_matters": "A blind fill would inject page type information, so missingness needs care.",
        },
    ]
)

limits

                   limitation_name  ...                                     why_it_matters
0  same-window snapshot limitation  ...  I can rank current opportunity, but I cannot c...
1    patterned keyword missingness  ...  A blind fill would inject page type informatio...

[2 rows x 3 columns]


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled - markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.